# Effect of temperature on flux and apparent permeability

The upstream pressure is held at `CONDITIONS['P_upstream']` and the temperature is swept over `CONDITIONS['T_range']` (both from `study_config.py`); the last section repeats the sweep at the low / mid / high of `P_range`.

Each point is labelled with its regime (the dominant path and the limiting step, `calculations/wall_labels.py`), and an Arrhenius law $J = A\,e^{-E_a/RT}$ is fitted over each regime's window: a straight line on $\ln J$ against $1/T$ whose slope gives $E_a$. The figures draw those **regime fits**. The table also gives the **single fit**, one Arrhenius line over every temperature: it averages over the regimes the sweep crosses, so it is reported but not drawn.

$\Phi_{app} = J\,L/\Delta\sqrt{P}$ ($\Delta P$ for L2a) comes from the same flux, at the same fixed pressure, so its activation energy is that of $J$.

How to read the figures and tables of all four sweeps: `docs/how-to/parameter-sweeps.md`.


## 1. Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
from scipy import stats
from calculations.config.model_config import (R, CONDITIONS, MICROSTRUCTURE,
                                              OXIDE_DEFECTS, build_simulation_config)
from calculations.oxide_permeation import (get_oxide_properties_at_T, get_metal_properties_at_T,
                                           molecular_diffusion_flux)
from calculations.permeation_calc import calculate_simple_metal_flux, calculate_defective_metal_flux
from calculations.interface_solver import calculate_oxide_metal_system
from calculations.parallel_oxide_defect_paths import (calculate_parallel_path_flux,
                                                      calculate_parallel_path_flux_defective_metal)
from calculations.wall_labels import label_model1, label_model2
from calculations.permeability import apparent_permeability, UNITS_SIEVERTS, UNITS_HENRY
import pandas as pd
from IPython.display import display
from calculations.figures import results_dir, save_panels
from calculations import sweep_figures as sf
FIGS = results_dir("effect-of-temperature")                             # figures and fit tables go in this folder

SIM          = build_simulation_config()
P_up, P_down = CONDITIONS['P_upstream'], SIM['P_downstream']
L_ox, L_m    = SIM['L_oxide'], SIM['L_metal']
T = np.linspace(*CONDITIONS['T_range'], CONDITIONS['n_points'])
x = 1000 / T

LEVELS = {   # level: (title, wall thickness for Φ_app)
    'L1':  ('Level 1: Perfect metal (lattice diffusion)',          L_m),
    'L2a': ('Level 2a: Perfect oxide alone (molecular H$_2$)',     L_ox),
    'L2b': ('Level 2b: Perfect oxide + perfect metal',             L_ox + L_m),
    'L3':  ('Level 3: Defective oxide + perfect metal',            L_ox + L_m),
    'L4':  ('Level 4: Defective metal (traps + grain boundaries)', L_m),
    'L5':  ('Level 5: Defective oxide + defective metal',          L_ox + L_m),
}

In [ ]:
def solve(level, Tk):
    """Flux and regime label for one level at one temperature."""
    if level.endswith('L6'):
        return solve_m2(level, Tk)                       # Model 2 levels, defined further down
    ox = get_oxide_properties_at_T(SIM['oxide_name'], Tk); ox['thickness'] = L_ox
    me = get_metal_properties_at_T(SIM['metal_name'], Tk); me['thickness'] = L_m
    D, K  = me['D_metal'], me['K_s_metal']
    micro = dict(microstructure_params=MICROSTRUCTURE, lattice_density=MICROSTRUCTURE['N_L'], mode='both')
    if level == 'L1':  return calculate_simple_metal_flux(D, K, L_m, P_up, P_down)['flux'], label_model1('L1')
    if level == 'L2a': return molecular_diffusion_flux(ox['D_ox'], ox['K_ox'], L_ox, P_up, P_down), label_model1('L2a')
    if level == 'L2b': r = calculate_oxide_metal_system(P_up, P_down, ox, me)
    if level == 'L3':  r = calculate_parallel_path_flux(P_up, P_down, ox, me, OXIDE_DEFECTS)
    if level == 'L4':  r = calculate_defective_metal_flux(D_lattice=D, K_s=K, metal_thickness=L_m,
                                                          P_up=P_up, P_down=P_down, temperature=Tk, **micro)
    if level == 'L5':  r = calculate_parallel_path_flux_defective_metal(P_up, P_down, ox, me,
                                                                        OXIDE_DEFECTS, Tk, **micro)
    return r.get('flux_total', r.get('flux')), label_model1(level, r, P_up, P_down, ox, me, OXIDE_DEFECTS, Tk, micro)

def sweep(level):
    J, regime = map(np.array, zip(*[solve(level, Tk) for Tk in T]))
    if level == 'L2a':                                   # Henry: J·L/ΔP
        return J, J * L_ox / (P_up - P_down), UNITS_HENRY, regime
    return J, apparent_permeability(J, P_up, P_down, LEVELS[level][1]), UNITS_SIEVERTS, regime

def arrhenius(x, y):
    fit = stats.linregress(x, np.log(y))
    return -fit.slope * R * 1000, np.exp(fit.intercept)   # Ea (J/mol), A


## 2: Physics (one solver call per level, then the sweep and the fit)

In [ ]:
# Figures: calculations/sweep_figures.py draws every figure of this notebook in the house style, one figure
# per panel (J and Φ_app), each written by calculations.figures.save_panels as <name>-<panel>.pdf, a 600 dpi
# .png and a _caption.tex whose numbers come from the results. Colour and marker mean the regime: colour = the
# limiting step, an open mark = a defect path carries most of the flux, a diamond = trap control
# (calculations/wall_labels.py). The fitted exponent of each regime is in the key below the plot.
from calculations.wall_labels import describe as look, fitted
ALLOY = sf.alloy_name(SIM['metal_name'])                # the alloy's name in the figure captions

## 3: Figures

In [ ]:
def p_tag(Pc):
    """Short pressure label: 1e-07 -> '1e-7 Pa', 1.0 -> '1e0 Pa', 1e+07 -> '1e7 Pa'."""
    return f'{Pc:.0e}'.replace('e+0', 'e').replace('e-0', 'e-') + ' Pa'

def fit_table(rows):
    """Every fitted line as one row: shown under the figure, and returned for the CSV."""
    table = pd.DataFrame(rows).replace({r'\$_2\$': '₂'}, regex=True)
    with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 220,
                           'display.max_columns', 20):
        display(table)
    return table

def single_fit_row(y, rows, quantity, A_units, tag):
    """The Arrhenius fit over every temperature: a row of the table, never drawn."""
    Ea, A = arrhenius(x, y)
    rows.append({'quantity': quantity, 'condition': tag, 'fit': 'single fit',
                 'T window (K)': f'{T.min():.0f}–{T.max():.0f}', 'points': len(T),
                 'E_a (kJ/mol)': Ea / 1e3, 'A': A, 'A units': A_units})

def regime_rows(y, regime, rows, quantity, A_units, tag):
    """One Arrhenius fit per regime, each a row of the table; the figures draw the same fits."""
    for reg in dict.fromkeys(regime):
        m = regime == reg
        if not fitted(reg) or m.sum() < 3:
            continue                                     # drawn, not fitted
        Ea, A = arrhenius(x[m], y[m])
        rows.append({'quantity': quantity, 'condition': tag, 'fit': look(reg)[0],
                     'T window (K)': f'{T[m].min():.0f}–{T[m].max():.0f}', 'points': int(m.sum()),
                     'E_a (kJ/mol)': Ea / 1e3, 'A': A, 'A units': A_units})

def arrhenius_figure(level, curves=None, pressures=None):
    """The Arrhenius figures of one level, J and Φ_app (one figure each, regime-resolved fits), and the fit
    table, which also holds the single fit over every temperature. curves = [(tag, sweep output), ...], one per
    condition, at the upstream pressures `pressures` (default: P_up). Returns (panels, fit table)."""
    curves = curves or [(p_tag(P_up), sweep(level))]
    pressures = pressures or [P_up] * len(curves)
    units = curves[0][1][2]
    rows = []
    for r, (qty, A_units) in enumerate((('J', 'mol/m^2/s'), ('Phi_app', units))):
        for tag, out in curves:                         # out = (J, Φ_app, units, regime)
            single_fit_row(out[r], rows, qty, A_units, tag)
        for tag, out in curves:
            regime_rows(out[r], out[3], rows, qty, A_units, tag)
    three = len(curves) > 1
    panels = sf.sweep_panels('temperature', level,
                             [sf.curve(sf.p_label(Pc) if three else tag, T, out[0], out[1], out[3], T, Pc, L_ox, L_m)
                              for (tag, out), Pc in zip(curves, pressures)],
                             units, P_down, variant='3P' if three else 'op', alloy=ALLOY)
    return sf.show(panels), fit_table(rows)

##  4: Plot

In [ ]:
for level in LEVELS:
    panels, fits = arrhenius_figure(level)
    save_panels(panels, f"{level}_arrhenius", FIGS, module=sf.MODULE)
    fits.to_csv(FIGS / f"{level}_arrhenius_fits.csv", index=False)

In [ ]:
# =============================================================================
# MODEL 2 (L6 family): dissociation on the gas-facing surface, atomic H in oxide and metal
# Reuses solve() -> sweep() -> arrhenius_figure() above; only the solver is new.
# =============================================================================
from calculations.surface_kinetics import (get_all_properties, solve_steady_state_flux_L1L6,
                                           solve_steady_state_flux_L2aL6, solve_steady_state_flux_direct,
                                           calculate_mixed_defect_flux_L6, calculate_defective_metal_flux_L6,
                                           calculate_full_model_flux_L346_v2)
from calculations.config.model_config import METALS, get_surface_kinetics_from_config

MICRO   = SIM['microstructure']
DEFECTS = SIM['defect_config']
SIEVERTS_PINHOLE = SIM['oxide_defects'].get('use_sieverts_pinhole', False)

LEVELS_M2 = {   # level: (title, wall thickness for Φ_app) -- every Model 2 level is Sieverts
    'L1L6':  ('Level 1+6: Perfect metal + surface kinetics',                     L_m),
    'L2aL6': ('Level 2a+6: Perfect oxide alone + surface kinetics',              L_ox),
    'L2bL6': ('Level 2b+6: Perfect oxide + perfect metal + surface kinetics',    L_ox + L_m),
    'L3L6':  ('Level 3+6: Defective oxide + perfect metal + surface kinetics',   L_ox + L_m),
    'L4L6':  ('Level 4+6: Perfect oxide + defective metal + surface kinetics',   L_ox + L_m),
    'L5L6':  ('Level 5+6: Defective oxide + defective metal + surface kinetics', L_ox + L_m),
}
LEVELS.update(LEVELS_M2)

def solve_m2(level, Tk):
    """Flux and regime label for one Model 2 level, e.g. 'oxide surface' or 'pinholes/metal diffusion'."""
    p  = get_all_properties(SIM['oxide_name'], SIM['metal_name'], Tk)       # oxide surface + transport
    sm = get_surface_kinetics_from_config(SIM['metal_name'], Tk, METALS)    # bare-metal surface
    k, Keq, Dox, Kox, Dm, Ksm = p['k_diss'], p['K_eq'], p['D_ox'], p['K_ox'], p['D_m'], p['K_s_m']
    pin  = {} if SIEVERTS_PINHOLE else dict(k_diss_metal=sm['k_diss'], K_eq_metal=sm['K_eq'])
    mode = MICRO.get('mode', 'both')
    if level == 'L1L6':  r = solve_steady_state_flux_L1L6(P_up, P_down, L_m, sm['k_diss'], sm['K_eq'], Dm, Ksm)
    if level == 'L2aL6': r = solve_steady_state_flux_L2aL6(P_up, P_down, k, Keq, Dox, Kox, L_ox)
    if level == 'L2bL6': r = solve_steady_state_flux_direct(P_up, P_down, L_m, k, Keq, Dox, Kox, L_ox, Dm, Ksm)
    if level == 'L3L6':  r = calculate_mixed_defect_flux_L6(P_up, P_down, L_m, k, Keq, Dox, Kox, L_ox,
                                                            Dm, Ksm, DEFECTS, **pin)
    if level == 'L4L6':  r = calculate_defective_metal_flux_L6(P_up, P_down, L_m, Tk, k, Keq, Dox, Kox, L_ox,
                                                               Dm, Ksm, microstructure_params=MICRO, mode=mode)
    if level == 'L5L6':  r = calculate_full_model_flux_L346_v2(P_up, P_down, L_m, Tk, k, Keq, Dox, Kox, L_ox,
                                                               Dm, Ksm, MICRO, DEFECTS, mode=mode, **pin)
    return r.get('J_total', r.get('J_ss', r.get('flux'))), label_model2(level, r, DEFECTS)

for level in LEVELS_M2:
    panels, fits = arrhenius_figure(level)
    save_panels(panels, f"{level}_arrhenius", FIGS, module=sf.MODULE)
    fits.to_csv(FIGS / f"{level}_arrhenius_fits.csv", index=False)

## Three pressures: the temperature sweep at the low / mid / high of `P_range`

In [ ]:
# =============================================================================
# THREE PRESSURES: the temperature sweep at the low / mid / high of P_range
# Same figures as above, with one curve per pressure. Colour and marker still mean the regime;
# the name at the end of each curve and the line style of its fits mean the pressure.
# =============================================================================
from contextlib import contextmanager

P_lo, P_hi = CONDITIONS['P_range']
P_SET = [P_lo, np.sqrt(P_lo * P_hi), P_hi]          # low / mid (log scale) / high: 1e-7, 1, 1e7 Pa

@contextmanager
def at_pressure(Pc):
    """Run solve() and sweep() above at another upstream pressure; P_up is restored afterwards."""
    global P_up
    saved, P_up = P_up, Pc
    try:
        yield
    finally:
        P_up = saved

for level in LEVELS:
    curves = []
    for Pc in P_SET:
        with at_pressure(Pc):
            curves.append((p_tag(Pc), sweep(level)))
    panels, fits = arrhenius_figure(level, curves, pressures=P_SET)
    save_panels(panels, f"{level}_arrhenius_3P", FIGS, module=sf.MODULE)
    fits.to_csv(FIGS / f"{level}_arrhenius_3P_fits.csv", index=False)